#### import libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense


#### Load the SMS Spam dataset

In [2]:
url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'text'])

print(df.head())
print(df['label'].value_counts())


  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...
label
ham     4825
spam     747
Name: count, dtype: int64


#### Encode labels and prepare texts

In [3]:
le = LabelEncoder()
df['label_enc'] = le.fit_transform(df['label'])  # ham=0, spam=1

texts = df['text'].astype(str).tolist()
labels = df['label_enc'].values


In [5]:
df

,label,text,label_enc
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1
5568,ham,Will ü b going to esplanade fr home?,0
5569,ham,"Pity, * was in mood for that. So...any other s...",0
5570,ham,The guy did some bitching but I acted like i'd...,0


#### Train/validation split

In [6]:
X_train_texts, X_val_texts, y_train, y_val = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)


#### Tokenize and pad sequences

In [7]:
vocab_size = 10000      # max number of words to keep
max_len = 50            # max sequence length
oov_token = "<OOV>"     #Out Of Vocabulary

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train_texts)

X_train_seq = tokenizer.texts_to_sequences(X_train_texts)
X_val_seq   = tokenizer.texts_to_sequences(X_val_texts)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_val_pad   = pad_sequences(X_val_seq,   maxlen=max_len, padding='post', truncating='post')


#### Build a supervised embedding model

In [8]:
embedding_dim = 16  # small, just to demonstrate

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()


G:\DS\ML for teens\venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ ?                           │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

#### Train the model (classification task)

In [9]:
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)


Epoch 1/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 8s 14ms/step - accuracy: 0.8602 - loss: 0.3967 - val_accuracy: 0.8664 - val_loss: 0.3351
Epoch 2/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8658 - loss: 0.3173 - val_accuracy: 0.8664 - val_loss: 0.3081
Epoch 3/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.8779 - loss: 0.2516 - val_accuracy: 0.9211 - val_loss: 0.2033
Epoch 4/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9551 - loss: 0.1459 - val_accuracy: 0.9686 - val_loss: 0.1227
Epoch 5/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9762 - loss: 0.0900 - val_accuracy: 0.9749 - val_loss: 0.0913
Epoch 6/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9812 - loss: 0.0662 - val_accuracy: 0.9785 - val_loss: 0.0746
Epoch 7/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9854 - loss: 0.0528 - val_accuracy: 0.9812 - val_loss: 0.0636
Epoch 8/10
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9881 - loss: 0.0439 - val_accuracy:

#### Extract and inspect learned embeddings

In [12]:
embedding_weights = model.layers[0].get_weights()[0]  # shape: (vocab_size, embedding_dim)
word_index = tokenizer.word_index

def get_embedding(word):
    idx = word_index.get(word)
    if idx is None or idx >= vocab_size:
        return None
    return embedding_weights[idx]

print("Embedding for 'love':\n", get_embedding('love'))
print("Embedding for 'like':\n", get_embedding('like'))
print("Embedding for 'hello':\n", get_embedding('hello'))


Embedding for 'love':
 [-0.35395634 -0.27602628  0.2739497  -0.27425867  0.19770908 -0.35640365
  0.3894583   0.3663123   0.23883815 -0.10127173  0.32978493 -0.32996836
 -0.31909052 -0.35710865 -0.27027488  0.42134276]
Embedding for 'like':
 [-0.34754348 -0.32241017  0.37006098 -0.28029838  0.13872589 -0.3658966
  0.32499227  0.38090035  0.28348926  0.02269308  0.3099956  -0.32258433
 -0.27742308 -0.34251952 -0.24599953  0.3234882 ]
Embedding for 'hello':
 [-0.08998138 -0.05783467 -0.01276168 -0.09663545  0.30661884 -0.02309724
  0.0922392   0.12188385  0.07258575 -0.19319138  0.02232097 -0.03394186
 -0.01655401 -0.12165618 -0.03545294  0.15739092]
